In [9]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.linear_model import LinearRegression

from scipy.special import expit

ROOT = Path("..")

PROCESSED_DATA = ROOT / "data" / "processed"

features = pd.read_csv(
    PROCESSED_DATA / "enso_features.csv",
    parse_dates=["Date"]
)

print(features.shape)

features.head()

(887, 27)


,Date,nino34,nino3,nino4,iod,soi,nino34_lag1,nino34_lag2,nino34_lag3,nino3_lag1,...,iod_lag3,soi_lag1,soi_lag2,soi_lag3,nino34_roll3,nino3_roll3,nino4_roll3,iod_roll3,soi_roll3,future_nino34
0,1951-04-01,-0.23,-0.21,-0.42,-0.513,-0.3,-0.38,-1.04,-1.30,-0.33,...,0.256,-0.1,0.9,1.5,-0.550000,-0.433333,-0.703333,-0.014333,0.166667,-0.01
1,1951-05-01,-0.01,-0.18,0.26,-0.138,-0.7,-0.23,-0.38,-1.04,-0.21,...,0.211,-0.3,-0.1,0.9,-0.206667,-0.240000,-0.246667,-0.130667,-0.366667,0.00
2,1951-06-01,0.00,0.04,0.08,-0.190,0.2,-0.01,-0.23,-0.38,-0.18,...,0.259,-0.7,-0.3,-0.1,-0.080000,-0.116667,-0.026667,-0.280333,-0.266667,0.30
3,1951-07-01,0.30,0.62,0.23,-0.220,-1.0,0.00,-0.01,-0.23,0.04,...,-0.513,0.2,-0.7,-0.3,0.096667,0.160000,0.190000,-0.182667,-0.500000,0.17
4,1951-08-01,0.17,0.41,-0.26,0.124,-0.2,0.30,0.00,-0.01,0.62,...,-0.138,-1.0,0.2,-0.7,0.156667,0.356667,0.016667,-0.095333,-0.333333,0.51


In [10]:
feature_cols = [

    "nino34",
    "nino3",
    "nino4",
    "iod",
    "soi",

    "nino34_lag1",
    "nino34_lag2",
    "nino34_lag3",

    "nino3_lag1",
    "nino3_lag2",
    "nino3_lag3",

    "nino4_lag1",
    "nino4_lag2",
    "nino4_lag3",

    "iod_lag1",
    "iod_lag2",
    "iod_lag3",

    "soi_lag1",
    "soi_lag2",
    "soi_lag3",

    "nino34_roll3",
    "nino3_roll3",
    "nino4_roll3",
    "iod_roll3",
    "soi_roll3"
]

In [13]:
X = features[feature_cols]

y = features["future_nino34"]

model = LinearRegression()

model.fit(X, y)

print("Training Complete")

features["predicted_anomaly"] = model.predict(X)

features[
    [
        "Date",
        "future_nino34",
        "predicted_anomaly"
    ]
].head()

Training Complete


,Date,future_nino34,predicted_anomaly
0,1951-04-01,-0.01,-0.103956
1,1951-05-01,0.00,0.258477
2,1951-06-01,0.30,0.050090
3,1951-07-01,0.17,0.562206
4,1951-08-01,0.51,0.066990


In [16]:
TRANSITION_WIDTH = 0.20


def anomaly_to_probabilities(anomaly):

    p_el = expit(
        (anomaly - 0.5) /
        TRANSITION_WIDTH
    )

    p_la = expit(
        (-0.5 - anomaly) /
        TRANSITION_WIDTH
    )

    p_neutral = max(
        0,
        1 - p_el - p_la
    )

    total = p_el + p_neutral + p_la

    p_el /= total
    p_neutral /= total
    p_la /= total

    confidence = max(
        p_el,
        p_neutral,
        p_la
    )

    return pd.Series({

        "enso_el_nino_probability": p_el,

        "enso_neutral_probability": p_neutral,

        "enso_la_nina_probability": p_la,

        "enso_forecast_confidence": confidence

    })

In [17]:
probabilities = features[
    "predicted_anomaly"
].apply(
    anomaly_to_probabilities
)

features = pd.concat(
    [
        features,
        probabilities
    ],
    axis=1
)

In [18]:
enso_vertical = features[
    [
        "Date",

        "predicted_anomaly",

        "enso_el_nino_probability",

        "enso_neutral_probability",

        "enso_la_nina_probability",

        "enso_forecast_confidence"
    ]
]

enso_vertical.head()

,Date,predicted_anomaly,enso_el_nino_probability,enso_neutral_probability,enso_la_nina_probability,enso_forecast_confidence
0,1951-04-01,-0.103956,0.046540,0.832164,0.121296,0.832164
1,1951-05-01,0.258477,0.230124,0.747832,0.022045,0.747832
2,1951-06-01,0.050090,0.095388,0.844550,0.060061,0.844550
3,1951-07-01,0.562206,0.577137,0.417950,0.004913,0.577137
4,1951-08-01,0.066990,0.102933,0.841602,0.055464,0.841602
